In [139]:
import pandas as pd

In [140]:
df=pd.read_csv("IMDB Dataset.csv")

In [141]:
df.shape

(50000, 2)

In [142]:
df.head()


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [143]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [144]:
df.drop_duplicates(inplace=True)

In [145]:
df.shape

(49582, 2)

# Pre-processing

### 1. Converting text to lower case
### 2. Remove URLs
### 3. Remove HTML tags
### 4. Remove Punctuations marks
### 5. Removing stop word
### 6. Stemming
### 7. Encoding sentiment
### 8. Vectorization (TF-IDF)

# 1.Converitng to Lowercase

In [146]:
df["review"]=df["review"].str.lower()

# 2.Removing the URLs

In [147]:
import re
# sample_text="abc is the word, abc"  #abc=>xyz

# new_text=re.sub("abc","xyz", sample_text)

In [148]:
def remove_urls(text):
    text=re.sub(r"http\S+", "", text) ##(pattern, replacement, string)
    return text

df["review"]=df["review"].apply(remove_urls)

## 3. Remove Punctuations

In [149]:
def remove_punctuations(text):
    text=re.sub(r"[^A-Za-z0-9\s]" , "", text) ##(pattern, replacement, string)
    return text

df["review"]=df["review"].apply(remove_punctuations)

## 4. Remove HTML tags


In [150]:
def remove_html(text):
    text=re.sub(r"<.*?>" , "", text) ##(
    return text

df["review"]=df["review"].apply(remove_html)

## 5. Removing stop word


In [151]:
import nltk

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to /Users/sks/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /Users/sks/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /Users/sks/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [152]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [153]:
# sample_text="I like coding in python!"
# tokens=word_tokenize(sample_text)

In [154]:
def remove_stopwords(text):
    tokens=word_tokenize(text)
    stop_words=stopwords.words("english")

    for word in tokens:
        if word in stop_words:
            text=text.replace(word, "")

    return text

df["review"]=df["review"].apply(remove_stopwords)

In [155]:
df.head()

,review,sentiment
0,e revewers nted wtchg 1 oz epode ll ho...,positive
1,wderful ltle producti br br filming techniqu...,positive
2,thought ths wderful wy spend tme o hot s...,positive
3,bsclly res fmly lttle boy jke thks res zom...,negative
4,petter mtte love time mey vully stunng fi...,positive


## 6. Stemming


In [156]:
#PortersStemming

from nltk.stem import PorterStemmer

In [157]:
def stemming(text):
    ps=PorterStemmer()
    stemmed_words=[]

    tokens=word_tokenize(text)
    for token in tokens:
        stemmed_token=ps.stem(token)
        stemmed_words.append(stemmed_token)

    return "".join(stemmed_words)

df["review"]=df["review"].apply(stemming)

In [158]:
df.head()

,review,sentiment
0,erevewntedwtchg1ozepodllhookyrghtexctlihppeneb...,positive
1,wderltleproductibrbrfilmtechniquunssumoldtimeb...,positive
2,thoughtthwderwyspendtmeohotsummerweekendsttngn...,positive
3,bscllirefmlilttleboyjkethkrezombclosetprentref...,negative
4,pettermttelovetimemeyvullistunngfilmwtchmrmtte...,positive


## 7. Encoding

In [159]:
from sklearn.preprocessing import LabelEncoder

le=LabelEncoder()

df["sentiment"]=le.fit_transform(df["sentiment"])



In [160]:
y=df["sentiment"]

In [161]:
y

0        1
1        1
2        1
3        0
4        1
        ..
49995    1
49996    0
49997    0
49998    0
49999    0
Name: sentiment, Length: 49582, dtype: int64

## 8. Vectorization

In [162]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf=TfidfVectorizer(max_features=5000)

X=tf.fit_transform(df["review"])

In [163]:
X

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 5008 stored elements and shape (49582, 5000)>

## Dataset and DataLoaders


In [164]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test=train_test_split(
    X,y, test_size=0.2, random_state=42
)

In [165]:
X_train.shape

(39665, 5000)

In [166]:
X_test.shape

(9917, 5000)

In [167]:
import torch
from torch.utils.data import TensorDataset, DataLoader

In [168]:
X_train=X_train.toarray()
X_test=X_test.toarray()

In [169]:
X_train

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(39665, 5000))

In [170]:
train_set=TensorDataset(

    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float()
)

test_set=TensorDataset(

    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()
)


In [171]:
train_loader=DataLoader(train_set, shuffle=True, batch_size=64)
test_loader=DataLoader(test_set, shuffle=True, batch_size=64)


## Building our RNN

In [172]:
import torch.nn as nn
import torch.optim as optim

In [173]:
class RNN(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1):
        super().__init__()

        self.hidden_size=hidden_size
        self.num_layers=num_layers

        #RNN layer
        self.rnn=nn.RNN(input_size, hidden_size, num_layers, batch_first=True)

        # fully connected layer
        self.fc=nn.Linear(hidden_size,1)

    def forward(self,x):
        #optional ==> shape(num of layers, batch size, hidden size)
        h0=torch.zeros(self.num_layers, x.size(0), self.hidden_size)

        out, _=self.rnn(x,h0)
        # 1st value= hidden state of al the timestsp => (batch, seq_len, hidden size)
        # 2nd value= final hidden state of last timestep

        out=self.fc(out[:, -1, :])
        return out

In [174]:
input_size=X_train.shape[1]

model=RNN(input_size)
criterion=nn.BCELoss()
optimizer=optim.Adam(model.parameters())

## Training RNN

In [175]:
# unsqueeze & squeeze

epochs=10
for epoch in range(epochs):
    model.train()

    for Xb, yb in train_loader:
        optimizer.zero_grad()

        Xb=Xb.unsqueeze(1) #add singleton directiosn
        
        outputs=model(Xb) #(batch_size,1)

        outputs=torch.sigmoid(outputs.squeeze()) #(batch_size,1)

        loss=criterion(outputs, yb) #loss computed
        loss.backward() # backprop
        optimizer.step() #weights update

    print(f"epoch={epoch+1}/{epochs} and loss={loss.item()}")

epoch=1/10 and loss=0.6954145431518555
epoch=2/10 and loss=0.6624826192855835
epoch=3/10 and loss=0.6183578968048096
epoch=4/10 and loss=0.6119619011878967
epoch=5/10 and loss=0.5965551137924194
epoch=6/10 and loss=0.6389763355255127
epoch=7/10 and loss=0.6182407736778259
epoch=8/10 and loss=0.6329622268676758
epoch=9/10 and loss=0.6114442944526672
epoch=10/10 and loss=0.5839953422546387


In [176]:
#Evalute

model.eval()

with torch.no_grad():

    correct_vals=0
    tot_vals=0

    for Xb, yb in test_loader:
        Xb=Xb.unsqueeze(1)

        outputs=model(Xb)
        predicted=(torch.sigmoid(outputs.squeeze()) > 0.5).float()

        tot_vals +=yb.size(0)
        correct_vals += (predicted==yb).sum().item()

    print(f"accuracy={correct_vals/tot_vals*100}")

accuracy=49.672279923363924
